# 🎬 openGO_video — Colab GPU 影片生成伺服器

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MazingerSam/openGO_video/blob/main/notebooks/colab_opengovideo.ipynb)

本筆記本把 Colab 變成 **本機 Agent 專用的遠端 GPU 伺服器**：

- 🧠 使用開源模型：**AnimateDiff**（文字→影片）、**SVD-XT**（圖片→影片）
- 📡 透過 **Cloudflare Tunnel** 產生免註冊的公開網址
- 🤖 本機以 `python -m opengovideo` 送出工作，MP4 自動下載回 `outputs/`

> ⚠️ **執行前**：選單「執行階段 ➔ 變更執行階段類型」選 **L4 GPU**（推薦）或 **T4 GPU**。

## 步驟 0：檢查 GPU 環境

In [1]:
!nvidia-smi

Sun Nov 15 00:04:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 步驟 1：取得專案並安裝套件

從 GitHub 複製本專案（若已存在則更新），再安裝 Colab 端依賴。

In [2]:
import os
import sys

REPO_URL = "https://github.com/MazingerSam/openGO_video.git"
WORKDIR = "/content/openGO_video"

if os.path.isdir(WORKDIR):
    !cd {WORKDIR} && git pull --ff-only -q
else:
    !git clone -q {REPO_URL} {WORKDIR}

!pip install -q -r {WORKDIR}/requirements-colab.txt

sys.path.insert(0, f"{WORKDIR}/src")
import opengovideo
print(f"openGO_video {opengovideo.__version__} 已就緒")

openGO_video 0.1.0 已就緒


## 步驟 2：啟動橋接伺服器

以下儲存格會：
1. 預載兩個模型（約 1~2 分鐘，可略過讓第一次生成時再載入）
2. 啟動 FastAPI（`0.0.0.0:8000`）
3. 建立 Cloudflare Tunnel 並印出公開網址

看到 `🔗 公開網址：https://xxxx.trycloudflare.com` 後，回到本機執行 `python connect.py` 貼上網址即可。

> 若不想預載模型，把 `warmup(...)` 那行註解掉。

In [3]:
import os
from google.colab import userdata

try:
    # 讀取 Colab 左側 🔑 欄位中設定的 HF_TOKEN
    hf_token = userdata.get('HF_TOKEN')
    os.environ["HF_TOKEN"] = hf_token
    print("✅ 成功載入 HF_TOKEN，已設定至環境變數中！")
except Exception as e:
    print("⚠️ 尚未在 Colab 鑰匙圈中設定 'HF_TOKEN' 金鑰。")
    print("請點擊左側 🔑 圖示，新增一個名為 'HF_TOKEN' 的金鑰，並將您的 Hugging Face Token 貼入。")

⚠️ 尚未在 Colab 鑰匙圈中設定 'HF_TOKEN' 金鑰。
請點擊左側 🔑 圖示，新增一個名為 'HF_TOKEN' 的金鑰，並將您的 Hugging Face Token 貼入。


In [ ]:
from opengovideo.pipeline import warmup
from opengovideo.server import app, launch

# 可註解以節省啟動時間
warmup(app.state.store.pipeline, ["animatediff-v1.5", "svd-xt"])

launch(port=8000, tunnel=True)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
The config attributes {'motion_activation_fn': 'geglu', 'motion_attention_bias': False, 'motion_cross_attention_dim': None} were passed to MotionAdapter, but are not expected and will be ignored. Please verify your config.json configuration file.
/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

There are modules in UNetMotionModel that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/520 [00:00<?, ?it/s]

🌐 正在建立 Cloudflare Tunnel…


INFO:     Started server process [8732]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


 * Running on https://horn-psp-comment-invention.trycloudflare.com
 * Traffic stats available on http://127.0.0.1:20241/metrics
✅ Colab GPU 伺服器已就緒
🔗 公開網址：https://horn-psp-comment-invention.trycloudflare.com
   請把這個網址貼到本機執行：python connect.py


## 補充：斷線後重啟

Colab 閒置斷線後，重新「執行階段 ➔ 全部執行」，再從最後一格複製新的 Tunnel 網址，在本機執行：

```powershell
python connect.py
```

即可繼續使用，不需要重新安裝任何東西。